In [1]:
! pip install transformers==4.35.2

In [2]:
# translator_lib_hstate.py
# transformers==4.35.2 전제 (DynamicCache 사용 안 함)

from dataclasses import dataclass
from typing import Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------
# KV pack/unpack helpers
# ---------------------------

@dataclass
class ModelKVSpec:
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_size: int  # n_heads * head_dim


def pack_past_key_values(
    past_key_values: Tuple[Tuple[torch.Tensor, torch.Tensor], ...]
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    HF GPT2 past_key_values:
      tuple(n_layers) of (k, v)
      k,v: [B, n_heads, S, head_dim]
    return:
      K,V: [B, S, L, hidden_size]  where hidden_size=n_heads*head_dim
    """
    keys: List[torch.Tensor] = []
    values: List[torch.Tensor] = []

    for (k, v) in past_key_values:
        k2 = k.permute(0, 2, 1, 3).contiguous().view(k.size(0), k.size(2), -1)
        v2 = v.permute(0, 2, 1, 3).contiguous().view(v.size(0), v.size(2), -1)
        keys.append(k2)
        values.append(v2)

    K = torch.stack(keys, dim=2)   # [B, S, L, D]
    V = torch.stack(values, dim=2) # [B, S, L, D]
    return K, V


def unpack_past_key_values(
    K: torch.Tensor,
    V: torch.Tensor,
    spec: ModelKVSpec
) -> Tuple[Tuple[torch.Tensor, torch.Tensor], ...]:
    """
    K,V: [B, S, L, hidden_size]
    return HF past_key_values tuple:
      k,v: [B, n_heads, S, head_dim]
    """
    B, S, L, D = K.shape
    assert L == spec.n_layers, (L, spec.n_layers)
    assert D == spec.hidden_size, (D, spec.hidden_size)

    out = []
    for layer in range(L):
        k = K[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        v = V[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        out.append((k, v))
    return tuple(out)


# ---------------------------
# Raw Hidden extraction & KV recomputation
# ---------------------------

@torch.no_grad()
def extract_layer_inputs_gpt2(model, input_ids, attention_mask=None, use_cache=True):
    """
    "진짜 hidden states" = 각 블록 입력 (ln_1 통과 전)만 뽑아서 반환.

    GPT-2 output_hidden_states=True 일 때:
      hidden_states는 길이 (n_layer + 1)
      hidden_states[l] 는 block l 로 들어가는 입력 hidden (raw)

    return:
      H_raw: [B,S,L,d_model]  (ln_1 이전)
      past_key_values: 모델이 생성한 원본 KV (debug/target용)
      hidden_states tuple (debug)
    """
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=use_cache,
        output_hidden_states=True,
        return_dict=True,
    )
    hs = out.hidden_states  # len = n_layer + 1
    L = model.config.n_layer

    H_layers = []
    for l in range(L):
        H_layers.append(hs[l])  # raw hidden into block l (pre-ln_1)

    H_raw = torch.stack(H_layers, dim=2)  # [B,S,L,d]
    return H_raw, out.past_key_values, hs


def recompute_kv_from_hidden_raw_gpt2(model, H_raw: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    raw hidden (ln_1 전)에서 KV 복구:
      K^L = ln_1^L(H_raw^L) @ Wk^L + bk^L
      V^L = ln_1^L(H_raw^L) @ Wv^L + bv^L

    GPT-2 attn.c_attn: (q,k,v) concat projection.
    return K,V: [B,S,L,d_model] (head-flatten)
    """
    B, S, L, D = H_raw.shape
    assert L == model.config.n_layer
    assert D == model.config.n_embd

    Ks, Vs = [], []
    d = D

    for l in range(L):
        block = model.transformer.h[l]
        attn = block.attn

        # apply ln_1 here (since H_raw is pre-ln)
        Hl = block.ln_1(H_raw[:, :, l, :])  # [B,S,d]

        W = attn.c_attn.weight  # Conv1D: [d, 3d]
        b = attn.c_attn.bias    # [3d]

        Wk = W[:, d:2*d]        # [d, d]
        Wv = W[:, 2*d:3*d]      # [d, d]
        bk = b[d:2*d]           # [d]
        bv = b[2*d:3*d]         # [d]

        Kl = Hl @ Wk + bk       # [B,S,d]
        Vl = Hl @ Wv + bv       # [B,S,d]
        Ks.append(Kl)
        Vs.append(Vl)

    K = torch.stack(Ks, dim=2)  # [B,S,L,d]
    V = torch.stack(Vs, dim=2)  # [B,S,L,d]
    return K, V


def cosine_sim_flat(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    a1 = a.detach().float().reshape(-1)
    b1 = b.detach().float().reshape(-1)
    denom = (a1.norm() * b1.norm()).clamp_min(eps)
    return float(torch.dot(a1, b1) / denom)


# ---------------------------
# Cross-attention translator on Raw Hidden States
# ---------------------------

class CrossAttnBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, int(d_model * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(d_model * mlp_ratio), d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mem: torch.Tensor) -> torch.Tensor:
        q = self.ln_q(x)
        kv = self.ln_kv(mem)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


class LocalToSharedAdapterH(nn.Module):
    """
    T[mi -> Σ] : H_raw [B,S,L,Di] -> H* [B,S,Q]
    """
    def __init__(self, n_layers: int, d_in: int, q_dim: int, d_model: int = 256, n_heads: int = 8,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.n_layers = n_layers
        self.d_in = d_in
        self.q_dim = q_dim
        self.d_model = d_model

        self.in_ln = nn.LayerNorm(d_in)
        self.in_proj = nn.Linear(d_in, d_model)
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj = nn.Linear(n_layers * d_model, q_dim)
        self.act = nn.GELU()

    def forward(self, H_raw: torch.Tensor) -> torch.Tensor:
        B, S, L, Di = H_raw.shape
        assert L == self.n_layers and Di == self.d_in

        h = self.act(self.in_proj(self.in_ln(H_raw)))  # [B,S,L,d_model]
        cur = h[:, :, 0, :]
        outs = []
        for l, blk in enumerate(self.blocks):
            cur = blk(cur, h[:, :, l, :])
            outs.append(cur)
        cat = torch.cat(outs, dim=-1)
        y = self.act(self.out_proj(self.out_ln(cat)))  # [B,S,Q]
        return y


class SharedToLocalAdapterH(nn.Module):
    """
    T[Σ -> mi] : H* [B,S,Q] -> H_raw [B,S,L,Di]
    """
    def __init__(self, n_layers: int, q_dim: int, d_out: int, d_model: int = 256, n_heads: int = 8,
                 mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        assert q_dim % n_layers == 0, f"q_dim({q_dim}) must be divisible by n_layers({n_layers})"
        self.n_layers = n_layers
        self.q_dim = q_dim
        self.d_out = d_out
        self.d_model = d_model
        self.q_per_layer = q_dim // n_layers

        self.in_ln = nn.LayerNorm(self.q_per_layer)
        self.in_proj = nn.Linear(self.q_per_layer, d_model)
        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj = nn.Linear(n_layers * d_model, n_layers * d_out)
        self.act = nn.GELU()

    def forward(self, Hs: torch.Tensor) -> torch.Tensor:
        B, S, Q = Hs.shape
        assert Q == self.q_dim
        x = Hs.view(B, S, self.n_layers, self.q_per_layer)
        h = self.act(self.in_proj(self.in_ln(x)))  # [B,S,L,d_model]

        cur = h[:, :, 0, :]
        outs = []
        for l, blk in enumerate(self.blocks):
            cur = blk(cur, h[:, :, l, :])
            outs.append(cur)

        cat = torch.cat(outs, dim=-1)
        y = self.act(self.out_proj(self.out_ln(cat)))  # [B,S,L*d_out]
        return y.view(B, S, self.n_layers, self.d_out)


class SharedSpaceHTranslator(nn.Module):
    """
    Raw hidden states 기반: A(H_raw) <-> Σ <-> B(H_raw)
    Loss는 "번역된 raw hidden -> (ln_1 + Wk/Wv)로 KV 복구" 후 KV MSE로 계산
    """
    def __init__(
        self,
        a_layers: int, a_hidden: int,
        b_layers: int, b_hidden: int,
        q_dim: int = 1536,
        d_model: int = 256,
        n_heads: int = 8,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.A_to_S = LocalToSharedAdapterH(a_layers, a_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_A = SharedToLocalAdapterH(a_layers, q_dim, a_hidden, d_model, n_heads, dropout=dropout)

        self.B_to_S = LocalToSharedAdapterH(b_layers, b_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_B = SharedToLocalAdapterH(b_layers, q_dim, b_hidden, d_model, n_heads, dropout=dropout)

    def translate_A_to_B(self, H_A_raw: torch.Tensor) -> torch.Tensor:
        Hs = self.A_to_S(H_A_raw)
        H_B_raw = self.S_to_B(Hs)
        return H_B_raw

    def translate_B_to_A(self, H_B_raw: torch.Tensor) -> torch.Tensor:
        Hs = self.B_to_S(H_B_raw)
        H_A_raw = self.S_to_A(Hs)
        return H_A_raw

In [3]:
# code1_train_hstate_translator.py
# 목표: Raw HiddenStates Translator 학습 + (ln_1 + Wk/Wv)로 KV recompute 후 reconstruction loss
# - A=gpt2, B=gpt2-medium
# - WikiText 2000 step
# - 매 100 step마다 validation loss 출력
#
# transformers==4.35.2 전제, argparse 금지

import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


# -------------------
# 설정
# -------------------
MODEL_A_NAME = "gpt2"
MODEL_B_NAME = "gpt2-medium"

SEED = 42
CONTEXT_LEN = 64
BATCH_SIZE = 2
TRAIN_STEPS = 1000
VAL_EVERY = 100
VAL_BATCHES = 10

LR = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

Q_DIM = 1536  # 12,24 모두 나눠짐
D_MODEL = 256
N_HEADS = 8
DROPOUT = 0.0

SAVE_PATH = "hstate_translator_ckpt.pt"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_lm_blocks(tokenizer, texts, block_size: int):
    joined = "\n\n".join([t for t in texts if t is not None])
    ids = tokenizer(joined, return_tensors=None)["input_ids"]
    n_blocks = len(ids) // block_size
    ids = ids[: n_blocks * block_size]
    return [ids[i * block_size : (i + 1) * block_size] for i in range(n_blocks)]


def collate_fn(batch):
    input_ids = torch.tensor(batch, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask}


def recon_loss_via_kv(model_target, H_translated_raw, K_target, V_target):
    # raw hidden -> (ln_1 + Wk/Wv)로 KV 복구 -> 원본 KV와 비교
    K_hat, V_hat = recompute_kv_from_hidden_raw_gpt2(model_target, H_translated_raw)
    return torch.mean((K_hat - K_target) ** 2) + torch.mean((V_hat - V_target) ** 2)


@torch.no_grad()
def eval_loss(translator, modelA, modelB, val_loader, device):
    translator.eval()
    total, n = 0.0, 0

    for i, batch in enumerate(val_loader):
        if i >= VAL_BATCHES:
            break
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        H_A_raw, pastA, _ = extract_layer_inputs_gpt2(modelA, input_ids, attention_mask, use_cache=True)
        H_B_raw, pastB, _ = extract_layer_inputs_gpt2(modelB, input_ids, attention_mask, use_cache=True)

        K_A, V_A = pack_past_key_values(pastA)
        K_B, V_B = pack_past_key_values(pastB)

        H_A_to_B_raw = translator.translate_A_to_B(H_A_raw)
        loss_A2B = recon_loss_via_kv(modelB, H_A_to_B_raw, K_B, V_B)

        H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)
        loss_B2A = recon_loss_via_kv(modelA, H_B_to_A_raw, K_A, V_A)

        loss = loss_A2B + loss_B2A
        total += float(loss.item())
        n += 1

    translator.train()
    return total / max(1, n)


def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_A_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    ds = load_dataset("wikitext", "wikitext-2-raw-v1")
    train_blocks = make_lm_blocks(tokenizer, ds["train"]["text"], CONTEXT_LEN)
    val_blocks = make_lm_blocks(tokenizer, ds["validation"]["text"], CONTEXT_LEN)

    train_loader = DataLoader(train_blocks, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_blocks, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=False)

    modelA = GPT2LMHeadModel.from_pretrained(MODEL_A_NAME).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(MODEL_B_NAME).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    for p in modelA.parameters():
        p.requires_grad_(False)
    for p in modelB.parameters():
        p.requires_grad_(False)

    translator = SharedSpaceHTranslator(
        a_layers=modelA.config.n_layer, a_hidden=modelA.config.n_embd,
        b_layers=modelB.config.n_layer, b_hidden=modelB.config.n_embd,
        q_dim=Q_DIM, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT
    ).to(device).train()

    opt = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_iter = iter(train_loader)

    for step in range(1, TRAIN_STEPS + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            H_A_raw, pastA, _ = extract_layer_inputs_gpt2(modelA, input_ids, attention_mask, use_cache=True)
            H_B_raw, pastB, _ = extract_layer_inputs_gpt2(modelB, input_ids, attention_mask, use_cache=True)

            K_A, V_A = pack_past_key_values(pastA)
            K_B, V_B = pack_past_key_values(pastB)

        opt.zero_grad(set_to_none=True)

        H_A_to_B_raw = translator.translate_A_to_B(H_A_raw)
        loss_A2B = recon_loss_via_kv(modelB, H_A_to_B_raw, K_B, V_B)

        H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)
        loss_B2A = recon_loss_via_kv(modelA, H_B_to_A_raw, K_A, V_A)

        loss = loss_A2B + loss_B2A
        loss.backward()
        nn.utils.clip_grad_norm_(translator.parameters(), GRAD_CLIP_NORM)
        opt.step()

        if step % 10 == 0:
            print(f"[train] step={step:4d} loss={loss.item():.6f}")

        if step % VAL_EVERY == 0:
            val_loss = eval_loss(translator, modelA, modelB, val_loader, device)
            print(f"[valid] step={step:4d} recon_loss={val_loss:.6f}")

    ckpt = {
        "translator_state": translator.state_dict(),
        "config": {
            "MODEL_A_NAME": MODEL_A_NAME,
            "MODEL_B_NAME": MODEL_B_NAME,
            "Q_DIM": Q_DIM,
            "D_MODEL": D_MODEL,
            "N_HEADS": N_HEADS,
            "DROPOUT": DROPOUT,
            "a_layers": modelA.config.n_layer,
            "a_hidden": modelA.config.n_embd,
            "b_layers": modelB.config.n_layer,
            "b_hidden": modelB.config.n_embd,
        },
    }
    torch.save(ckpt, SAVE_PATH)
    print("saved:", SAVE_PATH)


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[train] step=  10 loss=7.734007
[train] step=  20 loss=6.581631
[train] step=  30 loss=6.075698
[train] step=  40 loss=5.850258
[train] step=  50 loss=5.560966
[train] step=  60 loss=5.244193
[train] step=  70 loss=5.260735
[train] step=  80 loss=5.217541
[train] step=  90 loss=5.181371
[train] step= 100 loss=5.002864
[valid] step= 100 recon_loss=5.049345
[train] step= 110 loss=4.769604
[train] step= 120 loss=4.782174
[train] step= 130 loss=5.043772
[train] step= 140 loss=4.909846
[train] step= 150 loss=4.691706
[train] step= 160 loss=4.600919
[train] step= 170 loss=4.483156
[train] step= 180 loss=4.556327
[train] step= 190 loss=4.340396
[train] step= 200 loss=4.431291
[valid] step= 200 recon_loss=4.430480
[train] step= 210 loss=4.244593
[train] step= 220 loss=4.367769
[train] step= 230 loss=4.316959
[train] step= 240 loss=4.188974
[train] step= 250 loss=4.093411
[train] step= 260 loss=4.143208
[train] step= 270 loss=4.250230
[train] step= 280 loss=4.156898
[train] step= 290 loss=3.974

In [10]:
# code2_infer_hstate_translate.py
# 목표:
# - 학습된 Raw HiddenStates Translator 기반: raw hidden 번역 후 (ln_1 + Wk/Wv)로 KV recompute
# - round-trip KV vs original KV cosine
# - Prefix 2개에 대해 4가지 generate 비교
#
# transformers==4.35.2 전제, argparse 금지

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


CKPT_PATH = "hstate_translator_ckpt.pt"
MAX_NEW_TOKENS = 30

TARGET_PREFIX_TOKENS = 64  # <-- 요청사항: prefix 길이를 64 토큰으로 맞춤

PREFIXES = [
    "Seoul is the capital of",
    "Paris is the capital of",
]


@torch.no_grad()
def extend_prefix_to_len_with_model(model, prefix_ids: torch.Tensor, target_len: int) -> torch.Tensor:
    """
    prefix_ids를 model로 greedy-extend 해서 토큰 길이를 target_len으로 맞춘다.
    - prefix가 이미 target_len 이상이면 truncate
    - 동일 토큰 시퀀스를 A/B 모두에 입력하기 위해, 여기서는 modelA로만 확장
    """
    model.eval()
    if prefix_ids.size(1) >= target_len:
        return prefix_ids[:, :target_len]

    # 한 번에 prefix 전체를 넣어 cache와 마지막 logits를 얻음
    out = model(input_ids=prefix_ids, use_cache=True, return_dict=True)
    past = out.past_key_values
    generated = prefix_ids

    # out.logits는 prefix의 마지막 토큰 위치까지 포함
    logits = out.logits

    while generated.size(1) < target_len:
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # greedy
        generated = torch.cat([generated, next_id], dim=1)

        out = model(input_ids=next_id, past_key_values=past, use_cache=True, return_dict=True)
        past = out.past_key_values
        logits = out.logits

    return generated

@torch.no_grad()
def get_past_and_H_raw_excluding_last_token(model, input_ids: torch.Tensor):
    """
    past는 prefix[:-1]에 대해 만들고,
    generate의 첫 입력으로 prefix[-1]을 넣는 방식.
    """
    assert input_ids.size(1) >= 2
    prefix_excl_last = input_ids[:, :-1]
    H_raw, past, _ = extract_layer_inputs_gpt2(model, prefix_excl_last, attention_mask=None, use_cache=True)
    return H_raw, past


@torch.no_grad()
def greedy_generate_from_past(model, tokenizer, prefix_ids, past_excl_last, max_new_tokens: int):
    model.eval()
    input_ids = prefix_ids[:, -1:]
    generated = prefix_ids.clone()
    past = past_excl_last

    for _ in range(max_new_tokens):
        out = model(input_ids=input_ids, past_key_values=past, use_cache=True, return_dict=True)
        logits = out.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        generated = torch.cat([generated, next_id], dim=1)
        past = out.past_key_values
        input_ids = next_id

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    cfg = ckpt["config"]

    tokenizer = GPT2TokenizerFast.from_pretrained(cfg["MODEL_A_NAME"])
    tokenizer.pad_token = tokenizer.eos_token

    modelA = GPT2LMHeadModel.from_pretrained(cfg["MODEL_A_NAME"]).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(cfg["MODEL_B_NAME"]).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    translator = SharedSpaceHTranslator(
        a_layers=cfg["a_layers"], a_hidden=cfg["a_hidden"],
        b_layers=cfg["b_layers"], b_hidden=cfg["b_hidden"],
        q_dim=cfg["Q_DIM"], d_model=cfg["D_MODEL"], n_heads=cfg["N_HEADS"], dropout=cfg["DROPOUT"]
    ).to(device).eval()
    translator.load_state_dict(ckpt["translator_state"])

    specA = ModelKVSpec(
        n_layers=modelA.config.n_layer,
        n_heads=modelA.config.n_head,
        head_dim=modelA.config.n_embd // modelA.config.n_head,
        hidden_size=modelA.config.n_embd,
    )
    specB = ModelKVSpec(
        n_layers=modelB.config.n_layer,
        n_heads=modelB.config.n_head,
        head_dim=modelB.config.n_embd // modelB.config.n_head,
        hidden_size=modelB.config.n_embd,
    )

    for raw_prefix in PREFIXES:
        print("\n" + "=" * 80)
        print("RAW PREFIX:", raw_prefix)

        # 1) raw prefix -> ids
        prefix_ids = tokenizer(raw_prefix, return_tensors="pt").input_ids.to(device)

        # 2) prefix를 64 토큰으로 확장 (modelA로만 확장해서 동일 토큰 시퀀스를 만듦)
        # prefix_ids = extend_prefix_to_len_with_model(modelB, prefix_ids, TARGET_PREFIX_TOKENS)

        print(f"PREFIX TOKENS: {prefix_ids.size(1)} (target={TARGET_PREFIX_TOKENS})")
        # 너무 길면 출력이 지저분하니 앞부분만 확인용
        print("PREFIX (decoded):", tokenizer.decode(prefix_ids[0][:64], skip_special_tokens=True))

        # originals (prefix[:-1])
        H_A_raw, pastA = get_past_and_H_raw_excluding_last_token(modelA, prefix_ids)
        H_B_raw, pastB = get_past_and_H_raw_excluding_last_token(modelB, prefix_ids)

        K_A, V_A = pack_past_key_values(pastA)
        K_B, V_B = pack_past_key_values(pastB)

        # translate raw hidden
        with torch.no_grad():
            H_A_to_B_raw = translator.translate_A_to_B(H_A_raw)
            H_B_to_A_raw = translator.translate_B_to_A(H_B_raw)

            # round-trip raw hidden
            H_A_round_raw = translator.translate_B_to_A(H_A_to_B_raw)  # A->B->A
            H_B_round_raw = translator.translate_A_to_B(H_B_to_A_raw)  # B->A->B

        # recompute KV from translated raw hidden (includes ln_1 inside)
        K_A_to_B, V_A_to_B = recompute_kv_from_hidden_raw_gpt2(modelB, H_A_to_B_raw)
        K_B_to_A, V_B_to_A = recompute_kv_from_hidden_raw_gpt2(modelA, H_B_to_A_raw)

        # round-trip KV for cosine debug
        K_A_round, V_A_round = recompute_kv_from_hidden_raw_gpt2(modelA, H_A_round_raw)
        K_B_round, V_B_round = recompute_kv_from_hidden_raw_gpt2(modelB, H_B_round_raw)

        print(f"[cosine] A round-trip KV: key={cosine_sim_flat(K_A, K_A_round):.6f}, val={cosine_sim_flat(V_A, V_A_round):.6f}")
        print(f"[cosine] B round-trip KV: key={cosine_sim_flat(K_B, K_B_round):.6f}, val={cosine_sim_flat(V_B, V_B_round):.6f}")

        # build past_kv from translated KV
        pastA_from_B = unpack_past_key_values(K_B_to_A, V_B_to_A, specA)
        pastB_from_A = unpack_past_key_values(K_A_to_B, V_A_to_B, specB)

        # requested generations
        out1 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA,         MAX_NEW_TOKENS)  # A_original, model A
        out2 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB_from_A, MAX_NEW_TOKENS)  # A_to_B, model B
        out3 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA_from_B, MAX_NEW_TOKENS)  # B_to_A, model A
        out4 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB,         MAX_NEW_TOKENS)  # B_original, model B

        print("\n[generate] (1) A_original  -> model A")
        print(out1)
        print("\n[generate] (2) A_to_B      -> model B")
        print(out2)
        print("\n[generate] (3) B_to_A      -> model A")
        print(out3)
        print("\n[generate] (4) B_original  -> model B")
        print(out4)


if __name__ == "__main__":
    main()

device: cuda

RAW PREFIX: Seoul is the capital of
PREFIX TOKENS: 6 (target=64)
PREFIX (decoded): Seoul is the capital of
[cosine] A round-trip KV: key=0.405153, val=0.247499
[cosine] B round-trip KV: key=0.705629, val=0.219759

[generate] (1) A_original  -> model A
Seoul is the capital of South Korea, and is home to the country's largest and most famous tourist attraction.

The city is also home to the country's largest and

[generate] (2) A_to_B      -> model B
Seoul is the capital of the same name, but the name of the same name, and the name of the same name, and the name of the same name, and the

[generate] (3) B_to_A      -> model A
Seoul is the capital of its own.

The two-year-old boy who was killed in the same incident was also a victim of the same attack.



[generate] (4) B_original  -> model B
Seoul is the capital of South Korea, and is home to the country's largest concentration of foreign-born residents.

The city's population is estimated at around 1.

RAW PREFIX: Paris i